# ⚙️ Machine Guardian AI
## Predictive Maintenance: Predicting Machine Failure with Machine Learning

### 1. The idea — what are we building?

Machine Guardian AI is a beginner-friendly machine-learning project that explores whether sensor measurements from industrial equipment can help predict machine failure. Unexpected breakdowns can interrupt production, increase repair costs, delay work, and create safety concerns. Predictive maintenance uses data patterns to help maintenance teams identify equipment records that may deserve closer inspection.

The model learns from historical examples containing measurements such as air temperature, process temperature, rotational speed, torque, tool wear, and product type. It then predicts the class of a previously unseen record.

**Target prediction:** the `Machine failure` column.
- `0` = no machine failure recorded
- `1` = machine failure recorded

This is a **binary classification** task. The output is a learning demonstration and not a guarantee of future failure or a certified safety warning.

### 2. Models used

**Model A — Decision Tree Classifier:** learns a sequence of feature-based rules to classify a machine record as failure or no failure. It is comparatively easy to explain because its decisions are represented by tree branches.

**Model B — K-Nearest Neighbors (KNN) Classifier:** finds training records most similar to a new record and predicts a class from their labels. Numeric features are scaled first because KNN uses distances and the measurements have different units and ranges.

The notebook trains both models and compares them using accuracy, classification reports, and confusion matrices. The better model is determined from the results generated when the notebook runs, rather than assumed beforehand.

### 3. Why is it useful?

A machine-failure warning could help a maintenance team investigate equipment before relying only on fixed maintenance schedules. The project demonstrates a real industrial use of AI and teaches the complete beginner workflow: loading a dataset, checking data quality, preparing features and target labels, training models, and evaluating predictions.

Accuracy alone is not enough because failure records are uncommon. The notebook also reports precision and recall for the failure class and shows confusion matrices, which help identify missed failures and false alarms.

### 4. Dataset

**Dataset:** AI4I 2020 Predictive Maintenance Dataset from the UCI Machine Learning Repository. UCI describes it as a synthetic dataset reflecting industrial predictive-maintenance data, with 10,000 records and 14 columns.

- [Official UCI dataset page](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset)
- [Direct CSV file — can be downloaded and opened in Excel](https://raw.githubusercontent.com/somayaahmed27/AI4I-2020-Predictive-Maintenance-Dataset/main/AI4I%202020%20Predictive%20Maintenance%20Dataset.csv)

The dataset includes product type and sensor measurements such as temperature, rotational speed, torque, and tool wear. It also includes the target `Machine failure` and additional failure-mode indicators. Identifier columns and failure-mode indicators are excluded from the model inputs to prevent target leakage.

### 5. Data preparation

The notebook loads the dataset, displays its shape and sample rows, inspects columns and data types, checks missing values in every column, and counts duplicate records. The UCI description reports no missing values, but the code verifies this directly.

The `Machine failure` column becomes the target `y`; suitable sensor readings and product type become features `X`. The identifier columns `UDI` and `Product ID` are removed, as are the related failure-mode columns `TWF`, `HDF`, `PWF`, `OSF`, and `RNF`. These could leak information about the target into the input.

Numeric columns are prepared using a median imputer, while categorical values are prepared using the most frequent value and one-hot encoding. A stratified 80/20 train-test split is used. KNN receives standardized numeric features because distance calculations are sensitive to feature scale. All preprocessing is fitted within pipelines using the training data only.


## 1. Setup

Run this notebook from top to bottom. The first cell installs no packages; it uses common data-science libraries. If a library is missing in your environment, install it with `pip install pandas numpy matplotlib seaborn scikit-learn requests`.


In [ ]:
import io
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
sns.set_theme()
print("Libraries loaded successfully.")

## 6. Load the dataset

The CSV is a valid comma-separated file that can also be downloaded and opened in Excel: [https://raw.githubusercontent.com/somayaahmed27/AI4I-2020-Predictive-Maintenance-Dataset/main/AI4I%202020%20Predictive%20Maintenance%20Dataset.csv](https://raw.githubusercontent.com/somayaahmed27/AI4I-2020-Predictive-Maintenance-Dataset/main/AI4I%202020%20Predictive%20Maintenance%20Dataset.csv). The code below downloads the CSV directly, then reads it with pandas. An internet connection is needed.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/somayaahmed27/AI4I-2020-Predictive-Maintenance-Dataset/main/AI4I%202020%20Predictive%20Maintenance%20Dataset.csv"

response = requests.get(DATA_URL, timeout=30)
response.raise_for_status()

df = pd.read_csv(io.BytesIO(response.content))

print("Dataset shape:", df.shape)
display(df.head())

## 7. Inspect data quality and target balance

Check column names and data types, count missing values in each column, and check duplicate records. The UCI description reports no missing values for this dataset, but we verify it in code. Plot the target distribution to see how many records represent failures compared with non-failures.

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\\nMissing values per column:")
display(df.isna().sum().to_frame("missing_values"))

print("Duplicate rows:", df.duplicated().sum())

target_col = "Machine failure"
if target_col not in df.columns:
    raise KeyError(f"Expected target column {target_col!r} was not found.")

plt.figure(figsize=(5, 4))
sns.countplot(data=df, x=target_col)
plt.title("Target balance: Machine failure")
plt.xlabel("Machine failure (0 = No, 1 = Yes)")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

display(df[target_col].value_counts().rename(index={0: "No failure", 1: "Failure"}).to_frame("count"))

## 8. Prepare features and target

`Machine failure` is the target (`y`) that the model predicts. The sensor measurements and product type form the input features (`X`). Remove `UDI` and `Product ID` because they are identifiers, and remove `TWF`, `HDF`, `PWF`, `OSF`, and `RNF` because they are failure-mode labels closely related to the target. Excluding these columns helps prevent target leakage. A stratified train/test split keeps the rare failure class represented in both subsets.

In [ ]:
leakage_or_id_cols = [
    col for col in ["UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF"]
    if col in df.columns
]
X = df.drop(columns=[target_col] + leakage_or_id_cols)
y = df[target_col].astype(int)

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()

print("Excluded columns:", leakage_or_id_cols)
print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)
print("Target classes:", sorted(y.unique()))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Training records: {len(X_train):,}")
print(f"Testing records:  {len(X_test):,}")

## 9. Build the two named models

**Decision Tree Classifier** learns rule-like splits in the sensor data. **K-Nearest Neighbors (KNN) Classifier** predicts from similar training records, so numeric inputs are standardized for distance calculations. Preprocessing is placed in pipelines so transformations are learned from the training data only.

In [ ]:
def make_preprocessor(scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_pipe = Pipeline(numeric_steps)

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_features),
        ("categorical", categorical_pipe, categorical_features)
    ], remainder="drop")

models = {
    "Decision Tree": Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=False)),
        ("model", DecisionTreeClassifier(
            max_depth=8, class_weight="balanced", random_state=RANDOM_STATE
        ))
    ]),
    "KNN (k=7)": Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=True)),
        ("model", KNeighborsClassifier(n_neighbors=7, weights="distance"))
    ])
}

results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    results[name] = accuracy_score(y_test, y_pred)

comparison = pd.DataFrame({
    "Model": list(results.keys()),
    "Accuracy": list(results.values())
}).sort_values("Accuracy", ascending=False)

display(comparison.style.format({"Accuracy": "{:.2%}"}))

## 10. Evaluate results

Compare accuracy, precision, recall, F1-score, and confusion matrices. Accuracy is the overall share of correct predictions, but it can be misleading because failures are rare. Failure-class recall tells us how many actual failure records were detected; the confusion matrix also shows false alarms and missed failures.

In [ ]:
for name, y_pred in predictions.items():
    print("=" * 72)
    print(name)
    print(classification_report(
        y_test, y_pred,
        labels=[0, 1],
        target_names=["No failure", "Failure"],
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No failure", "Failure"]
    )
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()

## 11. Compare models and interpret the result

The bar chart compares test accuracy. The highest-accuracy model is reported automatically after the code runs. Also compare failure-class recall and confusion matrices before deciding which model is more useful for this problem.

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=comparison, x="Model", y="Accuracy")
plt.ylim(0, 1)
plt.ylabel("Test accuracy")
plt.title("Decision Tree vs KNN")
for i, row in enumerate(comparison.itertuples(index=False)):
    plt.text(i, row.Accuracy + 0.015, f"{row.Accuracy:.1%}", ha="center")
plt.tight_layout()
plt.show()

best_name = comparison.iloc[0]["Model"]
print(f"Highest test accuracy in this run: {best_name} ({comparison.iloc[0]['Accuracy']:.2%})")

## 12. Try a prediction

The following cell takes a held-out test record and asks the selected model to classify it as failure or no failure. This demonstrates how the trained model handles an unseen example. The result is educational and must not be used as a real industrial safety decision.

In [ ]:
# Choose an example row from the test set, then edit its values if desired.
example = X_test.iloc[[0]].copy()
display(example)

best_model = models[best_name]
example_prediction = int(best_model.predict(example)[0])

if hasattr(best_model, "predict_proba"):
    probabilities = best_model.predict_proba(example)[0]
    class_labels = best_model.named_steps["model"].classes_
    probability_map = dict(zip(class_labels, probabilities))
    failure_probability = float(probability_map.get(1, np.nan))
else:
    failure_probability = np.nan

print("Predicted result:", "POTENTIAL FAILURE" if example_prediction == 1 else "NO FAILURE PREDICTED")
if not np.isnan(failure_probability):
    print(f"Model-estimated failure probability: {failure_probability:.1%}")
print("Reminder: this is a learning demonstration, not an operational safety decision.")

## 13. Results and conclusion

Run all cells before writing your final results. The notebook calculates actual scores for this run, so do not insert guessed accuracy values.

In your conclusion, explain:
- Which model achieved higher test accuracy?
- Which model had better recall for the failure class?
- What do the confusion matrices reveal about false alarms and missed failures?
- Why is accuracy alone not enough when failure cases are rare?
- What would need to change before using such a model in a real factory?

**What this project demonstrates:** data loading and quality checks, feature/target preparation, leakage prevention, categorical encoding, scaling for KNN, model training, evaluation, and responsible interpretation.

**Limitations:** the dataset is synthetic and may not reflect every real machine. A real deployment would require representative operational data, validation over time, and review by maintenance and safety experts.

**Dataset references**
- UCI dataset page: https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset
- Direct CSV file: https://raw.githubusercontent.com/somayaahmed27/AI4I-2020-Predictive-Maintenance-Dataset/main/AI4I%202020%20Predictive%20Maintenance%20Dataset.csv